In [29]:
from astropy.cosmology import Planck18
%env XLA_PYTHON_CLIENT_ALLOCATOR=platform
import astropy.units as u
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
from glob import glob
import numpy as np
import sys
sys.path.append('../src/')
import jax.numpy as jnp
import matplotlib.pyplot as plt
from gwosc.api import fetch_event_json
import re
import os
import jax
import h5py
import pandas as pd
jax.local_devices()
jax.config.update("jax_enable_x64", True)


env: XLA_PYTHON_CLIENT_ALLOCATOR=platform


In [61]:

def get_event_far(file):
    match = re.search(r"(GW\d{6}_\d{6})", os.path.basename(file))
    if not match:
        return None

    event = match.group(1)

    try:
        evt_dict = fetch_event_json(event)
    except Exception as e:
        print(f"Failed to fetch {event}: {e}")
        return None

    try:
        evt_name = list(evt_dict["events"].keys())[0]
        data = evt_dict["events"][evt_name]
    except Exception as e:
        print(f"Malformed response for {event}: {e}")
        return None

    # Extract FAR
    far = data.get("far", data.get("far"))
    #far = 1 / (far_Hz * seconds_per_year)
    if far is None:
        print(f"No FAR found for {event}. Keys: {list(data.keys())}")
        return None

    return far, event

In [ ]:
data_paths = ['/mnt/home/ccalvk/ceph/GWTC-4/IGWN-GWTC4p0-*-combined_PEDataRelease.hdf5',]
#data_paths=['/mnt/home/ccalvk/ceph/GWTC-3/IGWN-GWTC3p0-v2-GW*_PEDataRelease_mixed_nocosmo.h5']
files = []
for path in data_paths:
    files += glob(path)

print(len(files))

86


In [134]:
fars = []
names = []  
for file in files:
    far, event = get_event_far(file)
    if far<1:
        fars.append(far)
        names.append(event)

In [135]:
event_ifars=pd.read_csv('/mnt/home/misi/src/Memory/results/prod_20260402c/auto_o3o4a_joint/event_ifars.txt', sep=' ')
event_ifars['far']=1/event_ifars['event_name']
print(len(np.where(event_ifars['far']<1)[0]))
events_TC4=event_ifars.loc[event_ifars['#'].str.startswith('GW2')]
good_evt_TC4=events_TC4.iloc[np.where(events_TC4['far']<1)]

148


In [136]:
only_TC4 = set(good_evt_TC4['#']).difference(names)
print((only_TC4)) #all just GWTC3 as it should be

{'GW200302_015811', 'GW200209_085452', 'GW200316_215756', 'GW200202_154313', 'GW200112_155838', 'GW200224_222234', 'GW200129_065458', 'GW200128_022011', 'GW200225_060421', 'GW200208_130117', 'GW200216_220804', 'GW200311_115853', 'GW200219_094415'}


In [137]:
not_TC4 = set(names).difference(good_evt_TC4['#'])
print((not_TC4)) # two low mass ones i need to cut

{'GW230529_181500', 'GW230518_125908'}


In [138]:
far_threshold=1
include=[]
fars=[]
for i, file in enumerate(files):
    samples=[]
    event_far, name = get_event_far(file)
    fars.append(event_far)

    with h5py.File(file, 'r') as f:
        if 'PublicationSamples' in f.keys():
            # O3a files
            samples = np.array(f['PublicationSamples']['posterior_samples'])
        elif 'C00:Mixed' in f.keys():
            # O3b files
            samples = np.array(f['C00:Mixed']['posterior_samples'])
        elif 'C00:NRSur7dq4' in f.keys(): #what waveform approximation did we use
            samples = np.array(f['C00:NRSur7dq4']['posterior_samples'])        
        elif 'C00:IMRPhenomNSBH' in f.keys(): #what waveform approximation did we use
            samples = np.array(f['C00:IMRPhenomNSBH']['posterior_samples'])   
        elif 'C00:IMRPhenomNSBH:LowSpin' in f.keys(): #what waveform approximation did we use
            samples = np.array(f['C00:IMRPhenomNSBH:LowSpin']['posterior_samples'])        
        else:   
            print(f"Available keys in file {name}: {list(f.keys())}")
            continue
    zs=samples['redshift'] [()]
    m1_det = samples['mass_1'][()]
    qs = samples['mass_ratio'][()]
    dLs = samples['luminosity_distance'][()] / 1e3

    # should this be reweighted? and if so, whats the prior here, bc this is combined not nocosmo
    m2s_det= m1_det * qs
    m2s_src= m2s_det / (1 + zs)
    prob = np.mean(m2s_src > 2.5)
    #df_det_chunk['m1d'] = df_det_chunk['m1'] * (1 + df_det_chunk['z']) 

    if prob !=1:
        print(name, prob)
    if event_far < far_threshold and prob>0.9:
        include.append(name)


GW230529_181500 0.0
GW230518_125908 0.0


In [139]:
file1 = open("../runs/INCLUDE_LIST.txt", "a")
for name in include:
    file1.write(name + "\n")
file1.close()